# Step 5 — Motor AI Model Development Pipeline
## CWRU Motor Bearing Vibration Fault Classification & Anomaly Detection

This notebook trains and evaluates multiple machine learning models across classification and unsupervised anomaly detection paradigms.

In [1]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import xgboost as xgb

# Load dataset
df = pd.read_csv('feature_time_48k_2048_load_1.csv')
features = ['rms', 'sd', 'kurtosis', 'crest', 'max', 'min', 'skewness']
X = df[features]
y_raw = df['fault']

le = LabelEncoder()
y_multi = le.fit_transform(y_raw)
y_binary = (y_raw != 'Normal_1').astype(int)

X_train, X_test, y_train_m, y_test_m, y_train_b, y_test_b = train_test_split(
    X, y_multi, y_binary, test_size=0.20, random_state=42, stratify=y_multi
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Data split: {X_train.shape[0]} train, {X_test.shape[0]} test samples.")

### Model Training & Comparative Evaluation Table

In [2]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, random_state=42),
    'Lightweight Neural Net (MLP)': MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train_m)
    t0 = time.time()
    y_pred_m = clf.predict(X_test_scaled)
    t_infer = (time.time() - t0) / len(X_test) * 1000
    y_pred_b = (y_pred_m != le.transform(['Normal_1'])[0]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_b, y_pred_b).ravel()
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_m, y_pred_m),
        'Precision': precision_score(y_test_b, y_pred_b),
        'Recall': recall_score(y_test_b, y_pred_b),
        'F1-Score': f1_score(y_test_b, y_pred_b),
        'False Alarms (FPR)': fp / (fp + tn),
        'Inference (ms)': t_infer
    })

pd.DataFrame(results)